# Prepare MLF1

Download and clean this ABS source, save its workbook and CSV, then publish its Delta table directly. Each notebook runs independently.

Use the job parameters described in [README](../README.md), and fill in `catalog` and `schema` in the final publishing cell.

Use only the underlying Data 1 sheet, not the presentation pivot. Keep the published ASGS geography and combined regions. Modelled identifies the source; it does not assert a seasonal-adjustment status.


## 1. Source settings

Edit these values when ABS changes this source. Review the workbook before updating the reference file. Each source keeps its own layout and parsing code.

In [ ]:
dataset = "MLF1"
dataset_key = "mlf1"
baseline_file = "MLF1.xlsx"
download_file = "MLF1.xlsx"
base_url = "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/"

# Use the underlying estimates rather than the presentation pivot; Excel row 4 contains headings.
data_sheet = "Data 1"
mlf_header_row = 3

## 2. Shared job parameters

The job supplies `release_month` (YYYY-MM), `run_date` (`{{job.start_time.iso_date}}`, the job start date in UTC), `job_run_id` (`{{job.run_id}}`) and `workspace_path` (the absolute `/Workspace/...` path to the project folder). Keep the reference workbook in its `Reference Datasets/` folder; downloaded workbooks and cleaned CSVs are saved under `outputs/runs/`. Install pandas, openpyxl and requests in the task environment.

In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
from IPython.display import display

dbutils.widgets.text("release_month", "", "ABS reference month (YYYY-MM)")
dbutils.widgets.text("run_date", "", "Job run date (YYYY-MM-DD, UTC)")
dbutils.widgets.text("job_run_id", "", "Job run ID")
dbutils.widgets.text("workspace_path", "", "Workspace project path")
release_month = dbutils.widgets.get("release_month")
run_date = dbutils.widgets.get("run_date")
job_run_id = dbutils.widgets.get("job_run_id")
workspace = Path(dbutils.widgets.get("workspace_path"))
download_month = pd.Timestamp(datetime.strptime(release_month, "%Y-%m"))
assert release_month == download_month.strftime("%Y-%m"), "Use YYYY-MM for release_month"
assert job_run_id and "{{" not in job_run_id, "Supply the resolved job run ID"
assert dbutils.widgets.get("workspace_path"), "Supply workspace_path"

# Separate job runs so a later download does not replace an earlier run's evidence.
baseline = workspace / "Reference Datasets"
run = workspace / "outputs" / "runs" / f"{run_date}_{job_run_id}" / dataset_key
print(dataset, release_month, run)

## 3. Download the release workbook

In [ ]:
source_url = base_url + f"{download_month:%b-%Y}/".lower() + download_file
response = requests.get(source_url, timeout=120, headers={"Cache-Control": "no-cache"})
response.raise_for_status()
assert response.content.startswith(b"PK"), f"{dataset}: download is not an XLSX file"

# Retain the original download so parsing changes can be checked against the source.
(run / "raw").mkdir(parents=True, exist_ok=True)
(run / "raw" / download_file).write_bytes(response.content)
print(source_url)

## 4. Read, check and reshape the workbook

Keep the existing structural and series-definition checks. The release-month check prevents this source from being saved under a different month.

In [ ]:
baseline_workbook = pd.ExcelFile(baseline / baseline_file)
downloaded_workbook = pd.ExcelFile(run / "raw" / download_file)

mlf_base = pd.read_excel(baseline_workbook, sheet_name=data_sheet, header=None)
mlf_raw = pd.read_excel(downloaded_workbook, sheet_name=data_sheet, header=None)
print("MLF1 sheets:", downloaded_workbook.sheet_names)
assert set(downloaded_workbook.sheet_names) == set(baseline_workbook.sheet_names), "MLF1: workbook sheets changed"
assert mlf_raw.iloc[1, 0] == mlf_base.iloc[1, 0], "MLF1: table identity changed"
mlf_base = mlf_base.iloc[mlf_header_row:].dropna(axis=1, how="all")
mlf_raw = mlf_raw.iloc[mlf_header_row:].dropna(axis=1, how="all")
# Headings encode units and geography vintage; a change needs review before positional parsing.
assert mlf_raw.iloc[0].tolist() == mlf_base.iloc[0].tolist(), "MLF1: column names, units, order or geography vintage changed"
mlf_base = mlf_base.iloc[1:].set_axis(mlf_base.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()
mlf_raw = mlf_raw.iloc[1:].set_axis(mlf_raw.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()

dimension_columns = mlf_raw.columns[1:4].tolist()
measure_columns = mlf_raw.columns[4:].tolist()
region_column = dimension_columns[2]
assert mlf_raw.notna().all().all(), "MLF1: missing dimension or estimate"
assert mlf_raw[measure_columns].dtypes.map(pd.api.types.is_numeric_dtype).all(), "MLF1: non-numeric observations"
# Keep the reference coverage; new or removed demographic or geographic categories need review.
for column in dimension_columns:
    assert set(mlf_raw[column]) == set(mlf_base[column]), f"MLF1: categories changed in {column}"
mlf_raw["Month"] = pd.to_datetime(mlf_raw["Month"])
rows_per_month = mlf_raw.groupby("Month").size()
# Unequal monthly coverage could make comparisons across sex, age and region misleading.
assert rows_per_month.nunique() == 1, "MLF1: months have different numbers of sex × age × region rows"
latest_month = mlf_raw["Month"].max()
assert latest_month == download_month, f"MLF1: latest month is not {download_month:%B %Y}"

print(mlf_raw.shape)
display(mlf_raw.head())

## 5. Tidy the output columns

In [ ]:
mlf = mlf_raw.rename(columns={"Month": "date", "Sex": "sex", "Age": "age", region_column: "region"}).melt(
    id_vars=["date", "sex", "age", "region"], value_vars=measure_columns, var_name="measure"
)
# Keep ABS region codes and combined-region labels as published; do not split or remap them.
mlf[["region_code", "region"]] = mlf["region"].str.extract(r"^(\d{3}) (.+)$")
assert mlf["region_code"].notna().all(), "MLF1: region-code format changed"
mlf["measure"] = mlf["measure"].str.replace(" ('000)", "", regex=False)
mlf["unit"] = "'000"
# Modelled identifies the source, not a seasonal-adjustment status.
mlf["series_type"] = "Modelled"
print(mlf.shape)
display(mlf[["date", "region_code", "region", "sex", "age", "measure", "value"]].head())

## 6. Save the cleaned CSV

Save the cleaned data in this job run's output folder. Rerunning the same source within the same job run overwrites its raw workbook and CSV.

In [ ]:
# Expose the published SA4 geography explicitly in the output column names.
df = mlf[["date", "measure", "sex", "age", "region_code", "region", "series_type", "unit", "value"]].rename(
    columns={"region_code": "sa4_code", "region": "sa4_region"}
)

# Recheck the release before export so the filename cannot mislabel the prepared data.
assert df["date"].notna().all(), f"{dataset}: missing observation dates"
assert df["date"].max() == download_month, f"{dataset}: latest month does not match {release_month}"

# Save the CSV before publishing so it remains available if the Delta write fails.
csv_file = run / f"{dataset_key}_{release_month}.csv"
df.to_csv(csv_file, index=False, date_format="%Y-%m-%d")
print(df.shape)
display(df.head())

## 7. Publish the Delta table

Replace `YOUR_CATALOG` and `YOUR_SCHEMA` with your existing destination names. `dataset_key` supplies this source's table name. Run this cell in Databricks after the CSV export.

The write creates a missing Delta table or overwrites its data. It uses the cleaned `df` already in memory; no staging Delta table or separate publisher is needed. This source updates independently of the other two. Plain overwrite does not automatically replace an existing table's schema.

In [ ]:
catalog = "YOUR_CATALOG"
schema = "YOUR_SCHEMA"
table = f"{catalog}.{schema}.{dataset_key}"

# Observation months are calendar dates; avoid introducing timestamps in the Delta table.
sdf = spark.createDataFrame(df)
sdf = sdf.withColumn("date", sdf["date"].cast("date"))

# Each workbook includes history, so overwrite instead of appending duplicate observations.
sdf.write.format("delta").mode("overwrite").saveAsTable(table)

print(f"Published {len(df):,} rows to {table}")